# 06 — Mixture of Experts

MoE scales model capacity without proportional compute by activating only a subset of parameters per token.

## Sparse MoE, Routing, and Load Balancing

**Mixture of Experts** (Shazeer et al., 2017; Fedus et al., 2021) replaces each dense FFN layer with *E* expert FFNs and a routing network:
$$y = \sum_{i=1}^{E} G_i(x) \cdot E_i(x)$$
where *G(x) = softmax(W_g x)* selects experts. **Sparse** MoE only activates the top-*k* experts (typically k=1 or k=2), so:
$$G(x) = \text{TopK}(\text{softmax}(W_g x + \epsilon), k)$$

**Capacity**: If each expert has capacity to process *C* tokens per batch, the total batch must route evenly. If tokens cluster to popular experts, some experts overflow (tokens are dropped) and others are underloaded.

**Load balancing losses** prevent routing collapse:
1. *Auxiliary loss* (Switch Transformer): encourages uniform routing by penalising the fraction of tokens routed to each expert
2. *z-loss* (ST-MoE): penalises large logits to prevent entropy collapse

**Expert parallelism**: experts are split across GPUs, so MoE models can use far more parameters than fit on one GPU while keeping per-device compute constant.

In [ ]:
# MoE transformer layer with top-k routing
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class ExpertFFN(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or d_model * 4
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
    def forward(self, x):
        return self.net(x)


class SparseRouter(nn.Module):
    def __init__(self, d_model, n_experts, top_k=2, aux_weight=0.01):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.aux_weight = aux_weight
        self.gate = nn.Linear(d_model, n_experts, bias=False)

    def forward(self, x):
        """
        x: (batch*seq_len, d_model) — all tokens flattened
        Returns: routing weights, expert indices, auxiliary loss
        """
        logits = self.gate(x)  # (N, E)
        probs = F.softmax(logits, dim=-1)

        # Top-k routing
        top_k_probs, top_k_idx = probs.topk(self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)  # renormalise

        # Auxiliary load-balancing loss (Switch Transformer)
        # f_i = fraction of tokens routed to expert i
        # P_i = mean routing probability for expert i
        # L_aux = n_experts * sum(f_i * P_i)
        one_hot = F.one_hot(top_k_idx[:, 0], self.n_experts).float()  # top-1 for load
        f_i = one_hot.mean(0)  # fraction per expert
        P_i = probs.mean(0)   # mean probability per expert
        aux_loss = self.n_experts * (f_i * P_i).sum()

        return top_k_probs, top_k_idx, aux_loss


class MoELayer(nn.Module):
    def __init__(self, d_model, n_experts=8, top_k=2, d_ff=None):
        super().__init__()
        self.experts = nn.ModuleList([ExpertFFN(d_model, d_ff) for _ in range(n_experts)])
        self.router = SparseRouter(d_model, n_experts, top_k)
        self.n_experts = n_experts
        self.top_k = top_k

    def forward(self, x):
        """
        x: (batch, seq, d_model)
        Returns: output, aux_loss
        """
        B, T, D = x.shape
        x_flat = x.reshape(B * T, D)

        probs, indices, aux_loss = self.router(x_flat)

        # Dispatch tokens to experts
        out = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e_idx in range(self.n_experts):
                mask = (indices[:, k] == e_idx)
                if mask.sum() == 0:
                    continue
                expert_input = x_flat[mask]
                expert_out = self.experts[e_idx](expert_input)
                out[mask] += probs[mask, k:k+1] * expert_out

        return out.reshape(B, T, D), aux_loss


torch.manual_seed(42)
moe = MoELayer(d_model=32, n_experts=8, top_k=2)
x = torch.randn(2, 16, 32)
y, aux = moe(x)
print(f'MoE output: {y.shape}')
print(f'Auxiliary load-balancing loss: {aux.item():.4f}')

# Parameter count
moe_params = sum(p.numel() for p in moe.parameters())
dense_params = sum(p.numel() for p in ExpertFFN(32).parameters())
print(f'Dense FFN params: {dense_params}')
print(f'MoE (8 experts) params: {moe_params}')
print(f'Capacity ratio: {moe_params/dense_params:.1f}x parameters, activating top-2/8 = 25%')

In [ ]:
# Load distribution analysis
import matplotlib.pyplot as plt

# Generate many batches and track expert utilisation
n_experts = 8
expert_counts = np.zeros(n_experts)

with torch.no_grad():
    for _ in range(100):
        x_batch = torch.randn(4, 32, 32)
        B, T, D = x_batch.shape
        x_flat = x_batch.reshape(B * T, D)
        _, indices, _ = moe.router(x_flat)
        for e in range(n_experts):
            expert_counts[e] += (indices[:, 0] == e).sum().item()

uniform = expert_counts.sum() / n_experts

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(n_experts), expert_counts, color='steelblue', alpha=0.8)
ax.axhline(uniform, color='tomato', linestyle='--', label=f'Uniform ({uniform:.0f})')
ax.set_xlabel('Expert ID')
ax.set_ylabel('Tokens processed')
ax.set_title('Expert Utilisation (should be ~uniform with aux loss)')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/moe_load.png', dpi=100, bbox_inches='tight')
plt.show()

load_imbalance = expert_counts.std() / expert_counts.mean()
print(f'Load imbalance (CV): {load_imbalance:.3f} (0=perfect balance)')